# Result analysis

## Setup

In [75]:
import pandas as pd
import numpy as np
import json

In [76]:
results_df = pd.read_csv("output/content_style_iou_results.csv")
with open('entities/artists_movements.json') as f:
    artists_movements = json.load(f)
with open('entities/objects_categories.json') as f:
    objects_categories = json.load(f)

In [77]:
results_df["template"] = results_df.apply(
    lambda row: row["prompt"]
        .replace(str(row["content_word"]), "<CONTENT>")
        .replace(str(row["style_word"]), "<STYLE>"),
    axis=1
)

In [78]:
artists_styles = [style for styles in artists_movements.values() for style in styles]
style_cols = []
for style in set(results_df["style_word"].unique().tolist() + artists_styles):
    if style in artists_movements:
        continue
    results_df[style] = results_df["style_word"].apply(lambda x: x == style or style in artists_movements.get(x, []))
    style_cols.append(style)

results_df["content_category"] = results_df["content_word"].apply(lambda x: objects_categories.get(x, x))

In [79]:
results_df.head(2)

,prompt,content_word,style_word,seed,use_quantile,iou_threshold,iou_baseline_mean,iou_baseline_std,support_a_baseline,support_b_baseline,...,Post Impressionism,Expressionism,Romanticism,Art Nouveau,Baroque,Surrealism,Impressionism,Naive Art Primitivism,Northern Renaissance,content_category
0,a painting of a person in the Abstract Express...,person,Abstract Expressionism,0,False,0.1,0.934055,0.055966,0.936863,0.988943,...,False,False,False,False,False,False,False,False,False,person
1,a painting of a person in the Abstract Express...,person,Abstract Expressionism,0,True,0.1,0.847405,0.038919,0.900000,0.900000,...,False,False,False,False,False,False,False,False,False,person


Some values are NaN because the prompt does not contain other tokens than the content and style tokens

In [80]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 43 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   prompt                            10800 non-null  object 
 1   content_word                      10800 non-null  object 
 2   style_word                        10800 non-null  object 
 3   seed                              10800 non-null  int64  
 4   use_quantile                      10800 non-null  bool   
 5   iou_threshold                     10800 non-null  float64
 6   iou_baseline_mean                 9000 non-null   float64
 7   iou_baseline_std                  9000 non-null   float64
 8   support_a_baseline                9000 non-null   float64
 9   support_b_baseline                9000 non-null   float64
 10  support_intersection_baseline     9000 non-null   float64
 11  support_union_baseline            9000 non-null   float64
 12  iou_

## Metrics

In [81]:
results_df["delta"] = results_df["iou_baseline_mean"] - results_df["iou_score"]
results_df["delta_cs"] = results_df["iou_baseline_cs_mean"] - results_df["iou_score"]
results_df["baseline_support_token"] = (results_df["support_a_baseline"]+ results_df["support_b_baseline"])/2
results_df["baseline_cs_support_token"] = (results_df["support_a_baseline_cs"]+ results_df["support_b_baseline_cs"])/2
results_df["support_content_style"] = (results_df["support_content"]+ results_df["support_style"])/2
results_df["threshold"] = results_df.apply(
    lambda row: f"{'P' if row['use_quantile'] else 'F'} {row['iou_threshold']}", axis=1
)

The computed metrics are expected to behave as follows:
- We expect the ``delta`` (difference between IoU scores) to be larger when the content and style heatmaps do not significantly intersect.
- ``iou_baseline_mean`` serves as a proxy for how much heatmap space, on average, each pair of words in the sentence shares.
- ``support_`` measures capture the relative number of pixels included in the heatmaps for:
- the first word (a or "content"),
- the second word (b or "style"),
- their union, and
- their intersection.

These metrics help us understand how many pixels are activated given a certain threshold. If support values drop too low, it’s more likely that the two heatmaps will share less salient space. While we do expect support to decrease with higher threshold values, it shouldn’t be excessively low. A reasonable estimate for expected support could be approximately $1 - t$, where $t$ is the threshold.

- We expect similar support values for both the baseline IoU and the content-style IoU scores, and ideally, they should be close to $1 - t$.
- Unlike the standard baseline, the content-style baseline (``iou_baseline_cs_mean``) considers only word pairs involving either the content or style word. This value should still be higher than the actual ``iou_score`` and indicates how much, on average, the content/style words overlap with other words in the prompt.
This measure is especially relevant for our case, as it focuses on the "important" words. If content and style words have little overlap with other words, it is more understandable that they might not overlap much with each other. This means that, if ``iou_baseline_cs_mean`` is close to ``iou_score``, we cannot confidently claim that the model is treating content and style as oppositional concepts.

In [82]:
group_cols = ["threshold"]

In [83]:
results_df.groupby(group_cols)["iou_score"].describe()

,count,mean,std,min,25%,50%,75%,max
threshold,,,,,,,,
F 0.1,600.0,0.948662,0.052058,0.458242,0.935448,0.963431,0.980397,0.999326
F 0.2,600.0,0.730891,0.169194,0.150988,0.650547,0.764071,0.858276,0.996265
F 0.3,600.0,0.439767,0.224661,0.009591,0.263950,0.423976,0.614165,0.974547
F 0.4,600.0,0.214534,0.202434,0.000000,0.044334,0.152306,0.340481,0.927709
F 0.5,600.0,0.096609,0.151514,0.000000,0.000468,0.020159,0.133317,0.858058
F 0.6,600.0,0.049973,0.114427,0.000000,0.000000,0.000000,0.030870,0.782859
F 0.7,600.0,0.029191,0.085070,0.000000,0.000000,0.000000,0.000830,0.665598
F 0.8,600.0,0.019009,0.070337,0.000000,0.000000,0.000000,0.000000,0.736039
F 0.9,600.0,0.017398,0.083459,0.000000,0.000000,0.000000,0.000000,0.811599


Given these skewed distribution, it might be better to consider the median.

Median ``delta`` and ``delta_cs`` values for each threshold configuration.

In [84]:
results_df.groupby(group_cols).agg(
    delta_med = ("delta", np.median),
    delta_cs_med = ("delta_cs", np.median),
    base_supp_med = ("baseline_support_token", np.median),
    base_cs_supp_med = ("baseline_cs_support_token", np.median),
    supp_content_style_med = ("support_content_style", np.median),
).round(3)

/tmp/ipykernel_47970/2818960735.py:1: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  results_df.groupby(group_cols).agg(
/tmp/ipykernel_47970/2818960735.py:1: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  results_df.groupby(group_cols).agg(


,delta_med,delta_cs_med,base_supp_med,base_cs_supp_med,supp_content_style_med
threshold,,,,,
F 0.1,0.014,0.014,0.985,0.988,0.981
F 0.2,0.093,0.102,0.914,0.926,0.878
F 0.3,0.224,0.242,0.791,0.803,0.701
F 0.4,0.310,0.292,0.650,0.652,0.533
F 0.5,0.255,0.252,0.489,0.501,0.418
F 0.6,0.167,0.158,0.322,0.332,0.265
F 0.7,0.097,0.086,0.175,0.178,0.127
F 0.8,0.048,0.031,0.062,0.063,0.043
F 0.9,0.016,0.002,0.009,0.009,0.007


Overall, delta values are positive, and this suggests that this model is considering the concepts of style and content as distinct, complementary qualities affecting the resulting image. Using a relative threshold computing quantiles lead to a higher support overall wrt considering fixed thresholds (this is true especiallu for higher fixed tresholds). 

Greater delta values are encountered in threshold values from 0.4 to 0.7. This is quite strange since we expect that the higher the threshold, the fewer the pixels involved in the IoU computations, thus the lower the probability of having overlapping regions. 

We choose 0.5 (percentile) and 0.4 (fixed) as good trade offs offering enough support to sustain our results and higher delta values.

In [85]:
filtered_results = results_df.loc[((results_df["iou_threshold"] == 0.5) & (results_df["use_quantile"] == True)) | 
                                  ((results_df["iou_threshold"] == 0.4) & (results_df["use_quantile"] == False))].copy()

## Median delta values considering prompt template and threshold configuration

In [90]:
filtered_results.groupby(["threshold","template"]).agg(
    delta_med = ("delta", np.median),
    delta_cs_med = ("delta_cs", np.median),
    base_cs_supp_med = ("baseline_cs_support_token", np.median),
).round(3)

/tmp/ipykernel_47970/800242297.py:1: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  filtered_results.groupby(["threshold","template"]).agg(
/tmp/ipykernel_47970/800242297.py:1: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  filtered_results.groupby(["threshold","template"]).agg(


delta_med  \
threshold template                                                    
F 0.4     <CONTENT> . <STYLE>                                 0.331   
          <CONTENT> <STYLE>                                     NaN   
          <CONTENT> in the <STYLE> style                      0.380   
          <CONTENT> with <STYLE> style                        0.317   
          a <STYLE> painting of a <CONTENT>                   0.240   
          a painting of a <CONTENT> in the <STYLE> style      0.238   
P 0.5     <CONTENT> . <STYLE>                                 0.219   
          <CONTENT> <STYLE>                                     NaN   
          <CONTENT> in the <STYLE> style                      0.259   
          <CONTENT> with <STYLE> style                        0.206   
          a <STYLE> painting of a <CONTENT>                   0.356   
          a painting of a <CONTENT> in the <STYLE> style      0.313   

                                                          delta_cs_med  \
threshold template                                                       
F 0.4     <CONTENT> . <STYLE>                                    0.367   
          <CONTENT> <STYLE>                                        NaN   
          <CONTENT> in the <STYLE> style                         0.416   
          <CONTENT> with <STYLE> style                           0.413   
          a <STYLE> painting of a <CONTENT>                      0.198   
          a painting of a <CONTENT> in the <STYLE> style         0.196   
P 0.5     <CONTENT> . <STYLE>                                    0.195   
          <CONTENT> <STYLE>                                        NaN   
          <CONTENT> in the <STYLE> style                         0.260   
          <CONTENT> with <STYLE> style                           0.285   
          a <STYLE> painting of a <CONTENT>                      0.255   
          a painting of a <CONTENT> in the <STYLE> style         0.269   

                                                          base_cs_supp_med  
threshold template                                                          
F 0.4     <CONTENT> . <STYLE>                                        0.721  
          <CONTENT> <STYLE>                                            NaN  
          <CONTENT> in the <STYLE> style                             0.717  
          <CONTENT> with <STYLE> style                               0.698  
          a <STYLE> painting of a <CONTENT>                          0.595  
          a painting of a <CONTENT> in the <STYLE> style             0.609  
P 0.5     <CONTENT> . <STYLE>                                        0.500  
          <CONTENT> <STYLE>                                            NaN  
          <CONTENT> in the <STYLE> style                             0.500  
          <CONTENT> with <STYLE> style                               0.500  
          a <STYLE> painting of a <CONTENT>                          0.500  
          a painting of a <CONTENT> in the <STYLE> style             0.500

Adding a medium seems to produce higher delta values for the P 0.5 configuration, while it is the opposite in the F 0.4 configuration. Why?

## Median delta values for each style

In [91]:
# Step 1: Melt the style columns into long format
melted_styles = filtered_results.melt(
    id_vars=[col for col in results_df.columns if col not in style_cols],
    value_vars=style_cols,
    var_name="style",
    value_name="is_style"
)

# Step 2: Keep only rows where the style is active (True)
active_styles = melted_styles[melted_styles["is_style"]]

# Step 3: Group by the style name
grouped = active_styles.groupby(["threshold","style"]).agg(
    delta_med=("delta", np.median),
    delta_cs_med=("delta_cs", np.median),
    base_cs_supp_med = ("baseline_cs_support_token", np.median),
).round(3)

grouped

/tmp/ipykernel_47970/2050980534.py:13: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  grouped = active_styles.groupby(["threshold","style"]).agg(
/tmp/ipykernel_47970/2050980534.py:13: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  grouped = active_styles.groupby(["threshold","style"]).agg(


delta_med  delta_cs_med  base_cs_supp_med
threshold style                                                            
F 0.4     Abstract Expressionism      0.269         0.297             0.625
          Action painting             0.382         0.401             0.695
          Analytical Cubism           0.309         0.241             0.505
          Art Nouveau                 0.369         0.392             0.653
          Baroque                     0.183         0.190             0.690
          Color Field Painting        0.396         0.418             0.637
          Contemporary Realism        0.427         0.426             0.735
          Cubism                      0.181         0.177             0.658
          Early Renaissance           0.332         0.317             0.647
          Expressionism               0.114         0.147             0.631
P 0.5     Abstract Expressionism      0.317         0.294             0.500
          Action painting             0.271         0.278             0.500
          Analytical Cubism           0.313         0.281             0.500
          Art Nouveau                 0.321         0.322             0.500
          Baroque                     0.018        -0.008             0.500
          Color Field Painting        0.305         0.306             0.500
          Contemporary Realism        0.271         0.259             0.500
          Cubism                      0.173         0.149             0.500
          Early Renaissance           0.300         0.307             0.500
          Expressionism               0.057        -0.014             0.500

The model is not behaving the same across styles. There are styles (Art Nouveau, Analytical Cubism) for which the distinction between the content heatmap and the style heatmap is much noticeable than others (Expressionism, Baroque).

## Median delta values for each content category

In [92]:
filtered_results.groupby(["threshold","content_category"]).agg(
    delta_med = ("delta", np.median),
    delta_cs_med = ("delta_cs", np.median),
    base_cs_supp_med = ("baseline_cs_support_token", np.median),
).round(3)

/tmp/ipykernel_47970/784908550.py:1: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  filtered_results.groupby(["threshold","content_category"]).agg(
/tmp/ipykernel_47970/784908550.py:1: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  filtered_results.groupby(["threshold","content_category"]).agg(


delta_med  delta_cs_med  base_cs_supp_med
threshold content_category                                           
F 0.4     accessory             0.249         0.205             0.773
          person                0.108         0.194             0.699
          vehicle               0.341         0.310             0.633
P 0.5     accessory             0.278         0.304             0.500
          person               -0.058        -0.080             0.500
          vehicle               0.265         0.267             0.500

In [97]:
filtered_results.groupby(["threshold","content_word"]).agg(
    delta_med = ("delta", np.median),
    delta_cs_med = ("delta_cs", np.median),
    base_cs_supp_med = ("baseline_cs_support_token", np.median),
).round(3)

/tmp/ipykernel_47970/570261315.py:1: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  filtered_results.groupby(["threshold","content_word"]).agg(
/tmp/ipykernel_47970/570261315.py:1: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  filtered_results.groupby(["threshold","content_word"]).agg(


delta_med  delta_cs_med  base_cs_supp_med
threshold content_word                                            
F 0.4     airplane           0.365         0.361             0.663
          bicycle            0.265         0.237             0.634
          boat               0.341         0.319             0.640
          bus                0.355         0.378             0.614
          car                0.325         0.311             0.626
          motorcycle         0.364         0.358             0.620
          person             0.108         0.194             0.699
          traffic light      0.249         0.205             0.773
          train              0.318         0.264             0.653
          truck              0.331         0.306             0.592
P 0.5     airplane           0.237         0.264             0.500
          bicycle            0.260         0.265             0.500
          boat               0.288         0.253             0.500
          bus                0.310         0.332             0.500
          car                0.193         0.201             0.500
          motorcycle         0.312         0.331             0.500
          person            -0.058        -0.080             0.500
          traffic light      0.278         0.304             0.500
          train              0.278         0.259             0.500
          truck              0.289         0.275             0.500

The model is still not behaving the same depending on the depicted content. Accessories and vehicles seems to have a higher distinction between content and styles than people.

## Median delta values for each style-content category pair

In [93]:
# Step 1: Melt the style columns into long format
melted_styles = filtered_results.melt(
    id_vars=[col for col in results_df.columns if col not in style_cols],
    value_vars=style_cols,
    var_name="style",
    value_name="is_style"
)

# Step 2: Keep only rows where the style is active (True)
active_styles = melted_styles[melted_styles["is_style"]]

# Step 3: Group by the style name
grouped = active_styles.groupby(["threshold","style", "content_category"]).agg(
    delta_med=("delta", np.median),
    delta_cs_med=("delta_cs", np.median),
    base_cs_supp_med = ("baseline_cs_support_token", np.median),
).round(3)

grouped

/tmp/ipykernel_47970/2543572231.py:13: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  grouped = active_styles.groupby(["threshold","style", "content_category"]).agg(
/tmp/ipykernel_47970/2543572231.py:13: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  grouped = active_styles.groupby(["threshold","style", "content_category"]).agg(


delta_med  delta_cs_med  \
threshold style                  content_category                            
F 0.4     Abstract Expressionism accessory             0.220         0.377   
                                 person                0.136         0.296   
                                 vehicle               0.291         0.296   
          Action painting        accessory             0.224         0.182   
                                 person                0.063         0.079   
                                 vehicle               0.405         0.423   
          Analytical Cubism      accessory             0.248         0.187   
                                 person                0.148         0.225   
                                 vehicle               0.319         0.242   
          Art Nouveau            accessory             0.261         0.207   
                                 person                0.167         0.257   
                                 vehicle               0.409         0.418   
          Baroque                accessory             0.298         0.202   
                                 person                0.175         0.306   
                                 vehicle               0.175         0.162   
          Color Field Painting   accessory             0.308         0.449   
                                 person                0.004         0.166   
                                 vehicle               0.413         0.422   
          Contemporary Realism   accessory             0.197         0.369   
                                 person               -0.054         0.053   
                                 vehicle               0.444         0.465   
          Cubism                 accessory             0.181         0.099   
                                 person                0.264         0.404   
                                 vehicle               0.179         0.224   
          Early Renaissance      accessory             0.138         0.272   
                                 person                0.230         0.354   
                                 vehicle               0.349         0.317   
          Expressionism          accessory             0.250         0.366   
                                 person                0.099         0.212   
                                 vehicle               0.098         0.099   
P 0.5     Abstract Expressionism accessory             0.319         0.425   
                                 person                0.056        -0.067   
                                 vehicle               0.324         0.314   
          Action painting        accessory             0.333         0.386   
                                 person               -0.210        -0.223   
                                 vehicle               0.293         0.285   
          Analytical Cubism      accessory             0.325         0.344   
                                 person               -0.084        -0.147   
                                 vehicle               0.315         0.285   
          Art Nouveau            accessory             0.224         0.325   
                                 person               -0.195        -0.238   
                                 vehicle               0.329         0.351   
          Baroque                accessory             0.244         0.163   
                                 person                0.115        -0.031   
                                 vehicle               0.006        -0.016   
          Color Field Painting   accessory             0.304         0.373   
                                 person               -0.067        -0.059   
                                 vehicle               0.318         0.310   
          Contemporary Realism   accessory             0.285         0.225   
                                 person               -0.199        -0.06

Reiterating previous comments, the "person" content is the one for which ``iou_score`` is lower than the baseline. 

## Experiments with highest and lowest delta values : examples for previous analysis

In [103]:
# Highest values of delta 
filtered_results.sort_values(by="delta", ascending=False)[["prompt", "template","content_word", "style_word", "threshold", "iou_score", "iou_baseline_mean", "iou_baseline_cs_mean", "delta", "delta_cs", "baseline_cs_support_token", "baseline_support_token", "support_content_style"]].head(10)

,prompt,template,content_word,style_word,threshold,iou_score,iou_baseline_mean,iou_baseline_cs_mean,delta,delta_cs,baseline_cs_support_token,baseline_support_token,support_content_style
996,a painting of a bus in the Color Field Paintin...,a painting of a <CONTENT> in the <STYLE> style,bus,Color Field Painting,F 0.4,0.001001,0.709303,0.476024,0.708302,0.475024,0.697187,0.816967,0.497553
834,a painting of a airplane in the Contemporary R...,a painting of a <CONTENT> in the <STYLE> style,airplane,Contemporary Realism,F 0.4,0.018057,0.669820,0.476385,0.651763,0.458327,0.682676,0.787095,0.508646
1194,a painting of a train in the Contemporary Real...,a painting of a <CONTENT> in the <STYLE> style,train,Contemporary Realism,F 0.4,0.064830,0.712081,0.494339,0.647251,0.429509,0.701495,0.802999,0.532321
474,a painting of a car in the Contemporary Realis...,a painting of a <CONTENT> in the <STYLE> style,car,Contemporary Realism,F 0.4,0.004652,0.626753,0.447886,0.622101,0.443234,0.632457,0.722276,0.482759
744,a painting of a airplane in the Action paintin...,a painting of a <CONTENT> in the <STYLE> style,airplane,Action painting,F 0.4,0.049469,0.657503,0.483761,0.608034,0.434292,0.662381,0.744969,0.524734
2709,a Abstract Expressionism painting of a bus,a <STYLE> painting of a <CONTENT>,bus,Abstract Expressionism,P 0.5,0.010570,0.614242,0.447238,0.603672,0.436667,0.500000,0.500000,0.500000
2814,a Contemporary Realism painting of a bus,a <STYLE> painting of a <CONTENT>,bus,Contemporary Realism,F 0.4,0.039788,0.634157,0.470304,0.594369,0.430515,0.622254,0.667019,0.517801
924,a painting of a bus in the Action painting style,a painting of a <CONTENT> in the <STYLE> style,bus,Action painting,F 0.4,0.064762,0.655890,0.488471,0.591128,0.423709,0.659924,0.736487,0.532319
456,a painting of a car in the Color Field Paintin...,a painting of a <CONTENT> in the <STYLE> style,car,Color Field Painting,F 0.4,0.003827,0.575844,0.427445,0.572017,0.423618,0.612055,0.695160,0.473547
2904,a Action painting painting of a train,a <STYLE> painting of a <CONTENT>,train,Action painting,F 0.4,0.048622,0.620577,0.476219,0.571955,0.427597,0.612367,0.655659,0.511351


In [110]:
# Lowest values of delta 
filtered_results[(~ filtered_results["delta"].isna())].sort_values(by="delta", ascending=False)[["prompt", "template", "content_word", "style_word", "threshold", "iou_score", "iou_baseline_mean", "iou_baseline_cs_mean", "delta", "delta_cs", "baseline_cs_support_token", "baseline_support_token", "support_content_style"]].tail(10)

,prompt,template,content_word,style_word,threshold,iou_score,iou_baseline_mean,iou_baseline_cs_mean,delta,delta_cs,baseline_cs_support_token,baseline_support_token,support_content_style
2940,a Art Nouveau painting of a train,a <STYLE> painting of a <CONTENT>,train,Art Nouveau,F 0.4,0.802634,0.443547,0.503847,-0.359087,-0.298787,0.728018,0.653747,0.901317
9153,person with Early Renaissance style,<CONTENT> with <STYLE> style,person,Early Renaissance,P 0.5,0.864521,0.455140,0.499617,-0.409381,-0.364905,0.500001,0.500000,0.500000
1932,a Cubism painting of a person,a <STYLE> painting of a <CONTENT>,person,Cubism,F 0.4,0.905572,0.494282,0.539337,-0.411290,-0.366234,0.764332,0.702856,0.907777
2832,a Cubism painting of a bus,a <STYLE> painting of a <CONTENT>,bus,Cubism,F 0.4,0.767436,0.355788,0.372184,-0.411649,-0.395252,0.672206,0.585996,0.873362
8091,airplane in the Expressionism style,<CONTENT> in the <STYLE> style,airplane,Expressionism,P 0.5,0.754751,0.342701,0.360157,-0.412050,-0.394594,0.500000,0.500000,0.500000
528,a painting of a car in the Expressionism style,a painting of a <CONTENT> in the <STYLE> style,car,Expressionism,F 0.4,0.824224,0.408480,0.077405,-0.415744,-0.746818,0.475277,0.629341,0.218503
8088,airplane in the Expressionism style,<CONTENT> in the <STYLE> style,airplane,Expressionism,F 0.4,0.927709,0.507743,0.553196,-0.419965,-0.374513,0.715523,0.688604,0.754229
5517,person . Contemporary Realism,<CONTENT> . <STYLE>,person,Contemporary Realism,P 0.5,0.672109,0.221813,0.223034,-0.450295,-0.449074,0.500000,0.500000,0.500000
6651,train . Expressionism,<CONTENT> . <STYLE>,train,Expressionism,P 0.5,0.598637,0.146179,0.139466,-0.452458,-0.459171,0.500001,0.500000,0.500001
168,a painting of a person in the Expressionism style,a painting of a <CONTENT> in the <STYLE> style,person,Expressionism,F 0.4,0.839966,0.266171,0.314815,-0.573794,-0.525151,0.621306,0.469061,0.875049


There is a particular 

## Highest delta values for content word and style word

In [112]:
filtered_results.groupby(["threshold", "content_word", "style_word"])[["delta","delta_cs", "iou_score"]].median().sort_values(by="delta", ascending=False).head(10)

delta  delta_cs  iou_score
threshold content_word style_word                                         
F 0.4     bus          Art Nouveau           0.522061  0.629856   0.022851
                       Contemporary Realism  0.489547  0.722704   0.038184
          motorcycle   Color Field Painting  0.482095  0.415650   0.004123
          airplane     Contemporary Realism  0.474421  0.669291   0.117628
          motorcycle   Action painting       0.470528  0.404718   0.113377
          truck        Art Nouveau           0.469737  0.415825   0.006159
          bus          Action painting       0.469241  0.628944   0.054029
          motorcycle   Art Nouveau           0.469222  0.547221   0.005692
          train        Action painting       0.468242  0.425294   0.264705
          motorcycle   Contemporary Realism  0.466181  0.423113   0.066485

In [113]:
filtered_results.groupby(["threshold", "content_word", "style_word"])[["delta","delta_cs", "iou_score"]].median().sort_values(by="delta", ascending=False).tail(10)

delta  delta_cs  iou_score
threshold content_word style_word                                         
F 0.4     car          Expressionism        -0.070850  0.035915   0.306751
P 0.5     person       Analytical Cubism    -0.083872 -0.147399   0.584247
          train        Expressionism        -0.086528 -0.107743   0.465848
          boat         Baroque              -0.097156 -0.134242   0.378401
          person       Expressionism        -0.141010 -0.152417   0.545950
                       Early Renaissance    -0.156790 -0.118368   0.585451
          car          Expressionism        -0.163817 -0.148176   0.437155
          person       Art Nouveau          -0.195474 -0.238332   0.617716
                       Contemporary Realism -0.199050 -0.067462   0.563153
                       Action painting      -0.210459 -0.222572   0.575319

As we said before, "person" is the content with higher overlap with "style". The most affected threshold configuration seems to be P 0.5.

## Tests for statistical significant difference between iou_score and iou_baseline_mean

Is iou_score significantly different from iou_baseline_mean on average across all prompts?

In [114]:
from scipy import stats

In [128]:
subset = results_df[(((results_df["iou_threshold"] == 0.5) & (results_df["use_quantile"] == True))) &
                    ((~results_df["iou_score"].isna()) & (~results_df["iou_baseline_mean"].isna()))]
t_stat, p_value = stats.ttest_rel(subset['iou_baseline_mean'], subset['iou_score'])
print(f"Paired t-test for IoU vs Baseline: t_stat={t_stat:.3f}, p_value={p_value:.03f}") 
t_stat, p_value = stats.ttest_rel(subset['iou_baseline_cs_mean'], subset['iou_score'])
print(f"Paired t-test for IoU vs CS Baseline: t_stat={t_stat:.3f}, p_value={p_value:.03f}") 
# p-value << 0.01 - Reject the null hypothesis of no difference

Paired t-test for IoU vs Baseline: t_stat=25.006, p_value=0.000
Paired t-test for IoU vs CS Baseline: t_stat=19.928, p_value=0.000


In [129]:
subset = results_df[(((results_df["iou_threshold"] == 0.4) & (results_df["use_quantile"] == False))) &
                    ((~results_df["iou_score"].isna()) & (~results_df["iou_baseline_mean"].isna()))]
t_stat, p_value = stats.ttest_rel(subset['iou_baseline_mean'], subset['iou_score'])
print(f"Paired t-test for IoU vs Baseline: t_stat={t_stat:.3f}, p_value={p_value:.03f}") 
t_stat, p_value = stats.ttest_rel(subset['iou_baseline_cs_mean'], subset['iou_score'])
print(f"Paired t-test for IoU vs CS Baseline: t_stat={t_stat:.3f}, p_value={p_value:.03f}") 
# p-value > 0.62 - Accept the null hypothesis of no difference

Paired t-test for IoU vs Baseline: t_stat=29.004, p_value=0.000
Paired t-test for IoU vs CS Baseline: t_stat=25.833, p_value=0.000


For both configurations and deltas, there is a statistical difference proving the strenght of our previous results.

In [131]:
# How many std away is the iou_score from the iou_baseline_mean?
results_df["z_score"] = (results_df["iou_baseline_mean"] - results_df["iou_score"]) / results_df["iou_baseline_std"]
results_df["z_score_cs"] = (results_df["iou_baseline_cs_mean"] - results_df["iou_score"]) / results_df["iou_baseline_std"]
results_df.groupby(group_cols).agg(
    delta_med = ("delta", np.median),
    delta_cs_med = ("delta_cs", np.median),
    base_supp_med = ("baseline_support_token", np.median),
    base_cs_supp_med = ("baseline_cs_support_token", np.median),
    supp_content_style_med = ("support_content_style", np.median),
    med_num_std = ("z_score", np.median),
    med_num_cs_std = ("z_score_cs", np.median),
).round(3)

/tmp/ipykernel_47970/2217583141.py:4: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  results_df.groupby(group_cols).agg(
/tmp/ipykernel_47970/2217583141.py:4: FutureWarning: The provided callable <function median at 0x795571fe3400> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  results_df.groupby(group_cols).agg(


,delta_med,delta_cs_med,base_supp_med,base_cs_supp_med,supp_content_style_med,med_num_std,med_num_cs_std
threshold,,,,,,,
F 0.1,0.014,0.014,0.985,0.988,0.981,1.179,1.206
F 0.2,0.093,0.102,0.914,0.926,0.878,1.293,1.509
F 0.3,0.224,0.242,0.791,0.803,0.701,1.412,1.621
F 0.4,0.310,0.292,0.650,0.652,0.533,1.393,1.435
F 0.5,0.255,0.252,0.489,0.501,0.418,1.169,1.148
F 0.6,0.167,0.158,0.322,0.332,0.265,0.947,0.915
F 0.7,0.097,0.086,0.175,0.178,0.127,0.736,0.664
F 0.8,0.048,0.031,0.062,0.063,0.043,0.569,0.478
F 0.9,0.016,0.002,0.009,0.009,0.007,0.431,0.151


Overall, ``iou_score`` are at ~ 1 standard deviation away (on the left side, given that it is smaller, and delta is positive) from ``iou_baseline_mean``and ``iou_baseline_cs_mean``.